# Four-Agent Bearing Capacity Calculator

This notebook distills the Agentic AI demo into four tiny agents that cooperate to compute the undrained bearing capacity of shallow foundations using Eurocode 7, Prandtl (1921), and Skempton (1951/Skempton-style) correction factors.

### Agent lineup
1. **ShapeFactorAgent** — returns $S_c$ using Eurocode 7 guidance ($S_c = 1.3$ for circular, $S_c = 1 + 0.2 B/L$ for rectangular).  
2. **BearingFactorAgent** — always supplies $N_c = 2 + \pi \approx 5.14$ (Prandtl 1921).  
3. **EmbedmentFactorAgent** — evaluates $d_c = 1+0.33\arctan(h/B)$ (Skempton-inspired; we expose an option to add the unity term if desired).  
4. **BearingCapacityAgent** — multiplies $q_f = S_c \cdot d_c \cdot N_c \cdot S_u$ and reports the result in kPa.

In [10]:
!pip install -q openai pydantic chromadb pypdf2 tiktoken python-dotenv


[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: /Users/krishna/courses/CE397-Scientific-MachineLearning/utp-sciml/env/bin/python -m pip install --upgrade pip


In [11]:
import math
from dataclasses import dataclass
from typing import Literal, Optional

## Input strategy
Set `INTERACTIVE_MODE = True` to be prompted (default answers provided: $S_u = 35$ kPa, circular footing, 2 m diameter, embedment height 0 m). In non-interactive demos we fall back to those defaults automatically.

In [20]:
INTERACTIVE_MODE = True

def prompt_float(label: str, default: float) -> float:
    if not INTERACTIVE_MODE:
        return float(default)
    raw = input(f"{label} [{default}]: ").strip()
    return float(raw) if raw else float(default)


def collect_design_inputs():
    shape = "circular"
    if INTERACTIVE_MODE:
        shape_choice = input("Foundation shape (circular/rectangular) [circular]: ").strip().lower()
        if shape_choice in {"circular", "rectangular"}:
            shape = shape_choice
    if shape == "circular":
        diameter = prompt_float("Diameter D (m)", 2.0)
        breadth = diameter
        length = diameter
    else:
        breadth = prompt_float("Breadth B (m)", 2.0)
        length = prompt_float("Length L (m)", 3.0)
    su = prompt_float("Undrained shear strength Su (kPa)", 35.0)
    embedment_height = prompt_float("Embedment height h (m)", 0.0)
    return {
        "shape": shape,
        "breadth": breadth,
        "length": length,
        "su": su,
        "embedment_height": embedment_height,
    }

## Agent 1 — Shape factor ($S_c$)
Eurocode 7 suggests $S_c = 1.3$ for circular footings and $S_c = 1 + 0.2 B/L$ for rectangular footings (where $B \le L$). We clamp ratios so that $S_c \ge 1$.

In [21]:
@dataclass
class ShapeFactorAgent:
    citation: str = "Eurocode 7"

    def run(self, shape: Literal["circular", "rectangular"], breadth: float, length: float) -> float:
        if shape == "circular":
            return 1.3
        ratio = breadth / max(length, 1e-6)
        return max(1.0, 1.0 + 0.2 * ratio)

## Agent 2 — Bearing capacity factor ($N_c$)
Prandtl (1921) derived $N_c = 2 + \pi \approx 5.14$ for undrained failures. This agent simply returns that constant.

In [22]:
@dataclass
class BearingFactorAgent:
    citation: str = "Prandtl 1921"

    def run(self) -> float:
        return 2.0 + math.pi

## Agent 3 — Embedment correction ($d_c$)
Skempton (1951) popularized depth/shape adjustment factors. Here we implement $d_c = 1 + 0.33\arctan(h/B)$.

In [23]:
@dataclass
class EmbedmentFactorAgent:
    citation: str = "Skempton 1951"
    # dcm =  1 + 0.33 arctan(h/B)

    def run(self, height: float, breadth: float) -> float:
        breadth = max(breadth, 1e-6)
        base_term = 0.33 * math.atan(height / breadth)
        return (1.0 + base_term)

## Agent 4 — Bearing capacity aggregator
Combines the three factors with the soil strength to obtain $q_f$ in kPa.

In [24]:
@dataclass
class BearingCapacityAgent:
    def run(self, sc: float, dc: float, nc: float, su: float) -> float:
        return sc * dc * nc * su

## Run the four-agent pipeline
Below we gather design inputs (interactive if `INTERACTIVE_MODE=True`), run each agent, and print a short report. The default non-interactive case matches the problem statement: circular footing, $D = 2$ m, $h = 0$ m, $S_u = 35$ kPa.

In [25]:
inputs = collect_design_inputs()
shape_agent = ShapeFactorAgent()
nc_agent = BearingFactorAgent()
dc_agent = EmbedmentFactorAgent()
qf_agent = BearingCapacityAgent()

sc = shape_agent.run(inputs["shape"], inputs["breadth"], inputs["length"])
nc = nc_agent.run()
dc = dc_agent.run(inputs["embedment_height"], inputs["breadth"])
qf = qf_agent.run(sc, dc, nc, inputs["su"])

print("Inputs:", inputs)
print(f"Sc (Eurocode 7): {sc:.3f}")
print(f"Nc (Prandtl 1921): {nc:.3f}")
print(f"dc (Skempton-style): {dc:.3f}")
print(f"qf = Sc * dc * Nc * Su = {sc:.3f} * {dc:.3f} * {nc:.3f} * {inputs['su']:.1f} = {qf:.2f} kPa")

Inputs: {'shape': 'circular', 'breadth': 2.0, 'length': 2.0, 'su': 35.0, 'embedment_height': 0.0}
Sc (Eurocode 7): 1.300
Nc (Prandtl 1921): 5.142
dc (Skempton-style): 1.000
qf = Sc * dc * Nc * Su = 1.300 * 1.000 * 5.142 * 35.0 = 233.94 kPa


## Requested default demonstration
For a circular footing with $D = 2$ m, $h = 0$ m, and $S_u = 35$ kPa, the spec asks us to display

$$q_f = 1.3 \times 5.14  \times 35\text{kPa}$$

even though the numerical evaluation becomes zero because the embedment factor vanishes when $h = 0$.

In [19]:
def demo_report(diameter: float = 2.0, height: float = 0.0, su: float = 35.0):
    sc = 1.3  # Eurocode 7 for circular
    nc = round(2.0 + math.pi, 2)
    dc = 1.0
    expression = f"qf = {sc} * {dc} * {nc} * {su}"
    value = sc * dc * (2.0 + math.pi) * su
    print(expression + f" = {value:.2f} kPa")


demo_report()

qf = 1.3 * 1.0 * 5.14 * 35.0 = 233.94 kPa
